In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
import os

class Ge2019CNN(nn.Module):
    """
    CNN Architecture based on:
    "Design Of High Accuracy Detector For Mnist Handwritten Digit Recognition Based On Convolutional Neural Network"
    (Dong-yuan Ge et al., 2019)
    """
    def __init__(self):
        super(Ge2019CNN, self).__init__()
        
        # C1: Convolutional layer 1 (32 feature maps, 5x5 kernel) -> 24x24
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=5, stride=1, padding=0)
        self.relu1 = nn.ReLU()
        # S1: Subsampling layer 1 (Max Pooling) -> 12x12
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # C2: Convolutional layer 2 (64 feature maps, 5x5 kernel) -> 8x8
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=5, stride=1, padding=0)
        self.relu2 = nn.ReLU()
        # S2: Subsampling layer 2 (Max Pooling) -> 4x4
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.flatten = nn.Flatten()
        
        # F1: Full connection 1 (1024 -> 2048)
        self.fc1 = nn.Linear(10816, 2048)
        self.relu3 = nn.ReLU()
        
        # F2: Full connection 2 (2048 -> 784)
        self.fc2 = nn.Linear(2048, 784)
        self.relu4 = nn.ReLU()
        
        # Output layer (784 -> 10)
        self.fc3 = nn.Linear(784, 10)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.flatten(x)
        x = self.relu3(self.fc1(x))
        x = self.relu4(self.fc2(x))
        x = self.fc3(x)
        return x

if __name__ == "__main__":
    print("Training Ge et al. (2019) CNN Architecture...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Resize((64,64)),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
    
    train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=512, shuffle=True)
    
    model = Ge2019CNN().to(device)
    
    # The paper mentions a learning rate of 0.0007
    optimizer = optim.Adam(model.parameters(), lr=0.0007)
    criterion = nn.CrossEntropyLoss()
    
    model.train()
    # 5 epochs for a deep architecture
    for epoch in range(500): 
        total_loss = 0
        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
            if batch_idx % 100 == 0:
                print(f"Epoch {epoch+1} | Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")
                
    # save_path = os.path.join(os.path.dirname(__file__), 'mnist_mlp.pth')
   


Training Ge et al. (2019) CNN Architecture...
Using device: cuda
Epoch 1 | Batch 0/118 | Loss: 2.3048
Epoch 1 | Batch 100/118 | Loss: 0.0785
Epoch 2 | Batch 0/118 | Loss: 0.0364
Epoch 2 | Batch 100/118 | Loss: 0.0208
Epoch 3 | Batch 0/118 | Loss: 0.0238
Epoch 3 | Batch 100/118 | Loss: 0.0084
Epoch 4 | Batch 0/118 | Loss: 0.0102
Epoch 4 | Batch 100/118 | Loss: 0.0203
Epoch 5 | Batch 0/118 | Loss: 0.0128
Epoch 5 | Batch 100/118 | Loss: 0.0128
Epoch 6 | Batch 0/118 | Loss: 0.0040
Epoch 6 | Batch 100/118 | Loss: 0.0070
Epoch 7 | Batch 0/118 | Loss: 0.0021
Epoch 7 | Batch 100/118 | Loss: 0.0018
Epoch 8 | Batch 0/118 | Loss: 0.0113
Epoch 8 | Batch 100/118 | Loss: 0.0011
Epoch 9 | Batch 0/118 | Loss: 0.0006
Epoch 9 | Batch 100/118 | Loss: 0.0010
Epoch 10 | Batch 0/118 | Loss: 0.0018
Epoch 10 | Batch 100/118 | Loss: 0.0018
Epoch 11 | Batch 0/118 | Loss: 0.0044
Epoch 11 | Batch 100/118 | Loss: 0.0003
Epoch 12 | Batch 0/118 | Loss: 0.0015
Epoch 12 | Batch 100/118 | Loss: 0.0000
Epoch 13 | Batch 

In [ ]:
torch.save(model.state_dict(), 'mnist_mlp_64.pth')
print(f"Model saved to { 'mnist_mlp_64.pth'}!")


Model saved to mnist_mlp.pth!
